In [1]:
import pandas as pd
import numpy as np
import scipy as sp
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
from statsmodels.stats.multitest import multipletests
from statsmodels.distributions.empirical_distribution import ECDF


In [2]:
island_data = pd.read_csv('srf_final_df.csv')
island_data = island_data.rename(columns={"Temperature (°C)": "temperature"})
island_data = island_data.drop(columns=["depth", "station", "gene_cluster"])
island_data = island_data.rename(columns={"abundance_sum": "abundance"})

In [3]:
# Get unique combinations of sample_ID and gene_id
sample_ids = island_data["sample_ID"].unique()
gene_ids = island_data["gene_id"].unique()
all_combinations = pd.MultiIndex.from_product([sample_ids, gene_ids], names=["sample_ID", "gene_id"]).to_frame().reset_index(drop=True)

# Get temperature/latitude info for each sample_ID
unique_samples = island_data.drop_duplicates(subset=["sample_ID"])[["sample_ID","temperature","latitude"]].set_index("sample_ID")

# Merge with original data
merged_df = pd.merge(all_combinations, island_data, on=["sample_ID","gene_id"], how="left")

# Fill in missing temperature/latitude from unique_samples
merged_df["temperature"] = merged_df.apply(
    lambda x: x["temperature"] if pd.notnull(x["temperature"]) else unique_samples.loc[x["sample_ID"], "temperature"],
    axis=1
)
merged_df["latitude"] = merged_df.apply(
    lambda x: x["latitude"] if pd.notnull(x["latitude"]) else unique_samples.loc[x["sample_ID"], "latitude"],
    axis=1
)

# Fill missing abundance_sum with 0
merged_df["abundance"] = merged_df["abundance"].fillna(0)

In [4]:
# Convert latitude to absolute value for correlation analysis
merged_df["abs_lat"] = merged_df["latitude"].abs()
merged_df = merged_df.drop(columns=["latitude"])

In [5]:
true_correlation = merged_df.groupby('gene_id').apply(
    lambda g: pd.Series(spearmanr(g['abundance'], g['abs_lat'], nan_policy='omit'),
                        index=['spearman_corr','pvalue'])
)

/var/folders/n2/1w3jxnnn76sggflcbz4b0pvr0000gn/T/ipykernel_76107/3326738231.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  true_correlation = merged_df.groupby('gene_id').apply(


In [6]:
adjusted_pvals = multipletests(true_correlation["pvalue"], alpha=0.05, method="fdr_bh")[1]
true_correlation["corrected_pvalue"] = adjusted_pvals

In [ ]:

true_correlation.to_csv("island_gene_correlation.csv")

In [ ]:
#this code and below is if you want to create your own null distirbution and generate more exact pvalues, in my case i did this but it didn't make anything more or less significant so I went with scipy spearmanr for ease of writing.
temp = []
for gene in gene_ids:
    gene_data = merged_df[merged_df["gene_id"] == gene]
    for i in range(1000):
        shuffled_abundance = np.random.permutation(gene_data["abundance"].values)
        corr_value, _ = spearmanr(shuffled_abundance, gene_data["abs_lat"])
        temp.append([gene, i, corr_value])


null_distribution = pd.DataFrame(temp, columns=["gene_id", "permutation", "spearman_r"])


In [ ]:

p_values = []
for gene in gene_ids:
    observed_corr = true_correlation.loc[gene, 'spearman_corr']
    dist = null_distribution.loc[null_distribution['gene_id'] == gene, 'spearman_r']
    cdf_func = ECDF(dist)
    cdf_val = cdf_func(observed_corr)
    pval = min(cdf_val, 1 - cdf_val)
    p_values.append([gene, observed_corr, pval])

p_value_df = pd.DataFrame(p_values, columns=['gene_id', 'observed_corr', 'pvalue'])




In [ ]:
p_corrected = multipletests(p_value_df["pvalue"], alpha=0.05, method="fdr_bh")[1]
p_value_df["pvalue_bonferroni"] = p_corrected


In [ ]:
# Get the unique genes from the null_distribution dataframe
genes = null_distribution['gene_id'].unique()
n_genes = len(genes)

# Define grid layout (e.g., 4 rows x 5 columns yields 20 subplots; we'll hide the extras)
rows, cols = 4, 5
fig, axes = plt.subplots(rows, cols, figsize=(15, 10))
axes = axes.flatten()

# Plot histogram of spearman_r values for each gene_id in its own subplot
for i, gene in enumerate(sorted(genes)):
    data = null_distribution.loc[null_distribution['gene_id'] == gene, 'spearman_r']
    sns.histplot(data, bins=30, kde=True, ax=axes[i])
    axes[i].set_title(gene)
    axes[i].set_xlabel("Spearman r")
    axes[i].set_ylabel("Frequency")

# Hide any extra subplots
for j in range(n_genes, rows * cols):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()